# Cross-Dataset Results: Baseline vs Improved Model

This notebook compares:
1. Baseline model (basic augmentation)
2. Improved model (robust augmentation)

Goal: Demonstrate that robust augmentation improves cross-dataset generalization.

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from training.train import train_model
from evaluation.evaluate import cross_dataset_evaluation
from utils.visualization import (
    plot_baseline_vs_improved,
    plot_cross_dataset_comparison
)

## 1. Train Improved Model

Train XceptionNet with **robust augmentation**.

In [ ]:
# Train improved model with robust augmentation
config_path = '../config/config.yaml'

print("Training improved model with ROBUST augmentation...")

# Uncomment to train (may take hours)
# train_model(
#     config_path=config_path,
#     augmentation_type='robust',
#     model_name='xception'
# )

## 2. Evaluate Improved Model

In [ ]:
# Evaluate improved model
improved_checkpoint = '../checkpoints/xception_robust/best_model.pth'

if Path(improved_checkpoint).exists():
    improved_results = cross_dataset_evaluation(
        checkpoint_path=improved_checkpoint,
        config_path=config_path,
        output_dir='../results/improved'
    )
else:
    print(f"Checkpoint not found at {improved_checkpoint}")

## 3. Load and Compare Results

In [ ]:
# Load baseline results
baseline_path = '../results/baseline/evaluation_results.json'
improved_path = '../results/improved/evaluation_results.json'

if Path(baseline_path).exists() and Path(improved_path).exists():
    with open(baseline_path, 'r') as f:
        baseline_results = json.load(f)
    
    with open(improved_path, 'r') as f:
        improved_results = json.load(f)
    
    # Create comparison dataframe
    comparison_data = []
    
    for dataset in baseline_results['metrics'].keys():
        baseline_metrics = baseline_results['metrics'][dataset]
        improved_metrics = improved_results['metrics'][dataset]
        
        comparison_data.append({
            'Dataset': dataset,
            'Baseline_Accuracy': baseline_metrics['accuracy'],
            'Improved_Accuracy': improved_metrics['accuracy'],
            'Improvement': improved_metrics['accuracy'] - baseline_metrics['accuracy'],
            'Baseline_F1': baseline_metrics['f1_score'],
            'Improved_F1': improved_metrics['f1_score'],
            'F1_Improvement': improved_metrics['f1_score'] - baseline_metrics['f1_score']
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print("\n" + "="*80)
    print("BASELINE VS IMPROVED COMPARISON")
    print("="*80)
    print(df_comparison.to_string(index=False))
else:
    print("Results files not found. Please train and evaluate both models first.")

## 4. Visualize Comparison

In [ ]:
if Path(baseline_path).exists() and Path(improved_path).exists():
    # For each test dataset, create comparison
    for dataset in baseline_results['metrics'].keys():
        plot_baseline_vs_improved(
            baseline_metrics=baseline_results['metrics'][dataset],
            improved_metrics=improved_results['metrics'][dataset],
            save_path=f'../results/comparison_{dataset.replace("+", "")}.png',
            title=f'Baseline vs Improved: {dataset}'
        )

## 5. Statistical Significance Testing

In [ ]:
if Path(baseline_path).exists() and Path(improved_path).exists():
    print("\n" + "="*80)
    print("IMPROVEMENT ANALYSIS")
    print("="*80)
    
    for _, row in df_comparison.iterrows():
        print(f"\n{row['Dataset']}:")
        print(f"  Accuracy: {row['Baseline_Accuracy']:.4f} → {row['Improved_Accuracy']:.4f}")
        print(f"  Improvement: {row['Improvement']:.4f} ({row['Improvement']/row['Baseline_Accuracy']*100:.2f}%)")
        print(f"  F1-Score: {row['Baseline_F1']:.4f} → {row['Improved_F1']:.4f}")
        print(f"  F1 Improvement: {row['F1_Improvement']:.4f} ({row['F1_Improvement']/row['Baseline_F1']*100:.2f}%)")

## 6. Create Thesis-Ready Tables

In [ ]:
if Path(baseline_path).exists() and Path(improved_path).exists():
    # Create comprehensive comparison table
    table_data = []
    
    for dataset in baseline_results['metrics'].keys():
        baseline = baseline_results['metrics'][dataset]
        improved = improved_results['metrics'][dataset]
        
        table_data.append({
            'Dataset': dataset,
            'Model': 'Baseline',
            'Accuracy': f"{baseline['accuracy']:.4f}",
            'Precision': f"{baseline['precision']:.4f}",
            'Recall': f"{baseline['recall']:.4f}",
            'F1-Score': f"{baseline['f1_score']:.4f}",
            'ROC-AUC': f"{baseline['roc_auc']:.4f}"
        })
        
        table_data.append({
            'Dataset': dataset,
            'Model': 'Improved',
            'Accuracy': f"{improved['accuracy']:.4f}",
            'Precision': f"{improved['precision']:.4f}",
            'Recall': f"{improved['recall']:.4f}",
            'F1-Score': f"{improved['f1_score']:.4f}",
            'ROC-AUC': f"{improved['roc_auc']:.4f}"
        })
    
    df_table = pd.DataFrame(table_data)
    
    print("\n" + "="*80)
    print("THESIS-READY PERFORMANCE TABLE")
    print("="*80)
    print(df_table.to_string(index=False))
    
    # Save to CSV for LaTeX inclusion
    df_table.to_csv('../results/performance_comparison_table.csv', index=False)
    print("\nTable saved to: ../results/performance_comparison_table.csv")

## 7. Conclusions

**Expected Findings:**
- Robust augmentation improves cross-dataset generalization
- Performance gap between intra-dataset and cross-dataset is reduced
- Improvement is more pronounced on cross-dataset scenarios

**Key Contributions:**
1. Demonstrated cross-dataset generalization limitation
2. Proposed robust augmentation as a solution
3. Quantified improvement across multiple datasets